In [ ]:
# Install Miniforge
!rm -rf /content/miniforge3

!wget -q \
  https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh \
  -O /content/miniforge.sh

!bash /content/miniforge.sh \
  -b \
  -p /content/miniforge3


In [ ]:
CONDA = "/content/miniforge3/bin/conda"

!{CONDA} config --system --remove-key channels 2>/dev/null || true
!{CONDA} config --system --add channels conda-forge
!{CONDA} config --system --add channels bioconda
!{CONDA} config --system --set channel_priority strict

!{CONDA} config --show channels
!{CONDA} config --show channel_priority

In [ ]:
!{CONDA} create \
  -y \
  -n meme_env \
  --override-channels \
  -c conda-forge \
  -c bioconda \
  python=3.10 \
  meme=5.5.9

In [ ]:
!{CONDA} run -n meme_env streme --version

In [ ]:
!{CONDA} run -n meme_env which streme

In [ ]:
import shutil
from google.colab import files
from pathlib import Path

### High-quality motif logo exports

The original STREME commands, protein inputs, motif-width limits (12-14), and
MEME version (5.5.9) are unchanged. Additional logos are generated from each
run's `streme.txt` using MEME Suite's native `ceqlogo` tool.

Each output ZIP includes a `high_quality_logos` directory with 600-dpi PNG,
vector EPS/PDF, an unmodified native EPS for comparison, and a motif-ID index.
Axis labels (`bits`, `Position`) and tick/position numbers use 1.5 times the
native font size. Adjust `LOGO_DPI` and `AXIS_FONT_SCALE` below if needed.
The logo canvas is 18 cm wide and 7 cm high. Original HTML results are preserved;
use the separate PNG/PDF/EPS files for figures. The bit heights are generated by
native ceqlogo defaults, including its pseudocount and small-sample correction,
and are not recalculated during font enlargement.

To regenerate figures from an existing run, complete the environment setup and
helper-definition cells, then call `export_sharp_logos(Path("/content/your_output"))`.
FASTA inputs alone require a STREME run first; this notebook does not upscale
existing low-resolution images.

References: [STREME](https://meme-suite.org/meme/doc/streme.html),
[ceqlogo](https://meme-suite.org/meme/doc/ceqlogo.html).


In [ ]:
# Ghostscript converts native EPS to high-resolution PNG and vector PDF.
!apt-get -qq update
!apt-get -qq install -y ghostscript


In [ ]:
import re
import subprocess
from pathlib import Path

# Appearance settings only; the STREME search parameters are unchanged.
LOGO_DPI = 600
AXIS_FONT_SCALE = 1.5


def enlarge_logo_axis_text(eps_text, scale=AXIS_FONT_SCALE):
    """Enlarge native MEME axis labels and position numbers, retaining logo data."""
    if scale <= 0:
        raise ValueError("Axis font scale must be positive")
    # In the MEME EPS template, fontsize controls the bits label, y ticks,
    # and sequence-position numbers. Letter heights are not recalculated.
    pattern = r"(?m)^(/fontsize\s+)([0-9]+(?:\.[0-9]+)?)(\s+def\b)"
    result, count = re.subn(
        pattern,
        lambda m: f"{m[1]}{float(m[2]) * scale:g}{m[3]}",
        eps_text,
    )
    if count != 1:
        raise ValueError("Unrecognized MEME EPS font definition; no font changes were applied")
    # Add room below the stacks for enlarged (rotated) position numbers.
    margin_pattern = r"(?m)^(/bottomMargin\s+)([0-9]+(?:\.[0-9]+)?)(\s+cm\s+def\b)"
    result, count = re.subn(
        margin_pattern,
        lambda m: f"{m[1]}{max(float(m[2]) * scale, 0.8):g}{m[3]}",
        result,
    )
    if count != 1:
        raise ValueError("Unrecognized MEME EPS bottom margin")
    return result


def export_sharp_logos(output_path):
    """Add native vector logos and 600-dpi PNGs to an existing STREME run."""
    output_path = Path(output_path).resolve()
    motif_file = output_path / "streme.txt"
    if not motif_file.is_file():
        raise FileNotFoundError(f"Run STREME first: {motif_file}")
    motif_ids = re.findall(r"(?m)^MOTIF\s+(\S+)", motif_file.read_text())
    if not motif_ids:
        print(f"No motifs found in {motif_file}; no logos to export.")
        return []

    logo_dir = output_path / "high_quality_logos"
    logo_dir.mkdir(exist_ok=True)
    # Export directly from STREME's motif matrices using the native MEME tool.
    # Each motif is selected by its 1-based index, avoiding filename/ID ambiguity.
    exported = []
    for index, motif_id in enumerate(motif_ids, 1):
        stem = f"motif_{index:03d}"
        raw_eps = logo_dir / f"{stem}_native.eps"
        subprocess.run([
            CONDA, "run", "--no-capture-output", "-n", "meme_env",
            "ceqlogo", "-i", str(motif_file), "-m", str(index),
            "-f", "EPS", "-w", "18", "-h", "7",
            "-x", "Position", "-y", "bits", "-d", "",
            "-o", str(raw_eps),
        ], check=True)
        eps = logo_dir / f"{stem}.eps"
        eps.write_text(enlarge_logo_axis_text(raw_eps.read_text()), encoding="utf-8")
        for device, suffix in (("pngalpha", ".png"), ("pdfwrite", ".pdf")):
            destination = logo_dir / f"{stem}{suffix}"
            subprocess.run([
                "gs", "-dSAFER", "-dBATCH", "-dNOPAUSE", "-dEPSCrop",
                f"-sDEVICE={device}", f"-r{LOGO_DPI}",
                *(["-dTextAlphaBits=4", "-dGraphicsAlphaBits=4"] if suffix == ".png" else []),
                f"-sOutputFile={destination}", str(eps),
            ], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        exported.append((stem, motif_id))

    import csv
    with (logo_dir / "motif_index.tsv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle, delimiter="\t")
        writer.writerow(["file_stem", "STREME_motif_ID"])
        writer.writerows(exported)
    (logo_dir / "README.txt").write_text(
        "Native MEME Suite logos generated from streme.txt.\n"
        f"PNG resolution: {LOGO_DPI} dpi. EPS and PDF are vector outputs.\n"
        f"Axis label and tick font scale: {AXIS_FONT_SCALE:g}x native.\n"
        "Position numbers are motif-relative, starting at 1.\n"
        "The STREME search settings and original HTML/text outputs are unchanged.\n"
        "Native ceqlogo defaults govern information content, pseudocounts, and sample correction.\n"
        "Files ending in _native.eps retain the unmodified ceqlogo drawing for comparison.\n"
        "The sharpened logos are separate exports; streme.html is not rewritten.\n",
        encoding="utf-8",
    )
    print(f"Exported {len(exported)} high-quality logos to {logo_dir}")
    return exported


STREME DBD

In [ ]:
!{CONDA} run -n meme_env streme \
    --protein \
    --p "/content/aor0-5_SM00066_domains_2.faa" \
    --o "/content/261101_streme_DBD_output" \
    --minw 12 \
    --maxw 14

In [ ]:
# ===== Create a ZIP archive of the output directory =====
output_path = Path("/content/261101_streme_DBD_output")

# Add publication-quality logos before packaging the original output.
export_sharp_logos(output_path)

zip_path = shutil.make_archive(
    "261101_streme_DBD_output",
    'zip',
    root_dir=output_path
)

print(f"ZIP file: {zip_path}")

# ===== Download =====
files.download(zip_path)

STREME non_DBD

In [ ]:
!{CONDA} run -n meme_env streme \
    --protein \
    --p "/content/aor0-5_non_SM00066_domains_AO090012000182_fixed.faa" \
    --o "/content/261101_streme_non_DBD_output" \
    --minw 12 \
    --maxw 14

In [ ]:
# ===== Create a ZIP archive of the output directory =====
output_path = Path("/content/261101_streme_non_DBD_output")

# Add publication-quality logos before packaging the original output.
export_sharp_logos(output_path)

zip_path = shutil.make_archive(
    "261101_streme_non_DBD_output",
    'zip',
    root_dir=output_path
)

print(f"ZIP file: {zip_path}")

# ===== Download =====
files.download(zip_path)

STREME FL

In [ ]:
!{CONDA} run -n meme_env streme \
    --protein \
    --p "/content/aor0-5_SM00066_FL.faa" \
    --o "/content/261101_streme_FL_output" \
    --minw 12 \
    --maxw 14

In [ ]:
# ===== Create a ZIP archive of the output directory =====
output_path = Path("/content/261101_streme_FL_output")

# Add publication-quality logos before packaging the original output.
export_sharp_logos(output_path)

zip_path = shutil.make_archive(
    "261101_streme_streme_FL_output",
    'zip',
    root_dir=output_path
)

print(f"ZIP file: {zip_path}")

# ===== Download =====
files.download(zip_path)